# COE 311K Final Project - Part 1 of 3

## Section 1: Introduction & Model Selection

## Section 2: Parameter Research & Justification

## Section 3: Numerical Methods Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import LogLocator

plt.rcParams.update({ #Got from Claude as an easy way to standardize all graphs
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 80
})


In [ ]:
L  = 0.02          # Inductance  [H] — spacecraft wiring
R  = 2.0           # Resistance  [Ω] — wiring + component resistance
C  = 50e-6         # Capacitance [F] — bus-stabilisation capacitor
V0 = 0.0           # Source voltage [V] — free-decay after switching event

q0 = 5.0e-3        # Initial charge  [C] — energy stored during operation
i0 = 0.0           # Initial current [A] — instantaneous switch opening

omega0 = 1.0 / np.sqrt(L * C) # Natural frequency  [rad/s]
alpha  = R / (2.0 * L) # Damping coefficient [rad/s]
omega_d = np.sqrt(omega0**2 - alpha**2) # Damped frequency   [rad/s]
zeta   = alpha / omega0 # Damping ratio      [—]


In [ ]:
def rlc_rhs(t, y):
    
    q, i = y
    dqdt = i
    didt = (V0 - R * i - q / C) / L     # Kirchhoff's voltage law: L*di/dt = V(t) - R*i - q/C
    return np.array([dqdt, didt])

def euler_forward(rhs, t_span, y0, h):
    
    t0, tf = t_span
    t = np.arange(t0, tf + h * 0.5, h)   # time grid (inclusive of tf)
    Y = np.zeros((len(t), len(y0)))
    Y[0] = y0

    for n in range(len(t) - 1):
        Y[n + 1] = Y[n] + h * rhs(t[n], Y[n]) # Single-slope update: y_{n+1} = y_n + h * f(t_n, y_n)

    return t, Y

def rk4(rhs, t_span, y0, h):
    
    t0, tf = t_span
    t = np.arange(t0, tf + h * 0.5, h)
    Y = np.zeros((len(t), len(y0)))
    Y[0] = y0

    for n in range(len(t) - 1):
        tn, yn = t[n], Y[n]
        k1 = rhs(tn,           yn)
        k2 = rhs(tn + h / 2,   yn + h / 2 * k1) # Four slope estimates
        k3 = rhs(tn + h / 2,   yn + h / 2 * k2)
        k4 = rhs(tn + h,       yn + h * k3)
        Y[n + 1] = yn + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4) # Weighted average: (k1 + 2k2 + 2k3 + k4) / 6

    return t, Y


## Section 4: Solutions & Comparison

In [ ]:
def analytical_solution(t):
    
    A = q0
    B = (i0 + alpha * q0) / omega_d
    q = np.exp(-alpha * t) * (A * np.cos(omega_d * t) + B * np.sin(omega_d * t))
    dq = np.exp(-alpha * t) * (
        (-alpha * A + omega_d * B) * np.cos(omega_d * t)
        + (-alpha * B - omega_d * A) * np.sin(omega_d * t)
    ) # Derivative for current
    return q, dq

t_span  = (0.0, 0.10)          # 100 ms — ~16 oscillation cycles
y_init  = np.array([q0, i0])   # [charge, current]
h_list  = [0.001, 0.0005, 0.0001]   # three step sizes

t_ref   = np.linspace(*t_span, 5000)
q_ref, i_ref = analytical_solution(t_ref)

euler_results = {} # Compute numerical solutions for all step sizes
rk4_results   = {}
for h in h_list:
    euler_results[h] = euler_forward(rlc_rhs, t_span, y_init, h)
    rk4_results[h]   = rk4(rlc_rhs,          t_span, y_init, h)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.0, 3.6))
fig.suptitle('Figure 1 — Charge q(t): Euler vs RK4 at Multiple Step Sizes', fontweight='bold')

ax = axes[0] # First axis — Euler Forward
method_name = "Euler's Forward"
results = euler_results

ax.plot(t_ref, q_ref * 1e3, 'k-', lw=2, label='Analytical', zorder=5)

h = h_list[0]
t_n, Y_n = results[h]
ax.plot(t_n, Y_n[:, 0] * 1e3, '--', color='orange', lw=1.5, label=f'h = {h:.4f} s')

h = h_list[1]
t_n, Y_n = results[h]
ax.plot(t_n, Y_n[:, 0] * 1e3, '--', color='red', lw=1.5, label=f'h = {h:.4f} s')

h = h_list[2]
t_n, Y_n = results[h]
ax.plot(t_n, Y_n[:, 0] * 1e3, '--', color='blue', lw=1.5, label=f'h = {h:.4f} s')

ax.set_ylabel('Charge q (mC)')
ax.set_title(f'{method_name} Method')
ax.legend(loc='upper right')
ax.grid(True)

ax = axes[1] # Second axis — RK4
method_name = "RK4"
results = rk4_results

ax.plot(t_ref, q_ref * 1e3, 'k-', lw=2, label='Analytical', zorder=5)

h = h_list[0]
t_n, Y_n = results[h]
ax.plot(t_n, Y_n[:, 0] * 1e3, '--', color='orange', lw=1.5, label=f'h = {h:.4f} s')

h = h_list[1]
t_n, Y_n = results[h]
ax.plot(t_n, Y_n[:, 0] * 1e3, '--', color='red', lw=1.5, label=f'h = {h:.4f} s')

h = h_list[2]
t_n, Y_n = results[h]
ax.plot(t_n, Y_n[:, 0] * 1e3, '--', color='blue', lw=1.5, label=f'h = {h:.4f} s')

ax.set_ylabel('Charge q (mC)')
ax.set_title(f'{method_name} Method')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.35)

axes[1].set_xlabel('Time (s)')
plt.show()
plt.close()


In [ ]:
h_fine = 0.0001
t_eu, Y_eu = euler_results[h_fine]
t_rk, Y_rk = rk4_results[h_fine]

fig, ax = plt.subplots(figsize=(4.0, 2.7))
ax.plot(q_ref  * 1e3, i_ref  * 1e3, 'k-',  lw=1.5,   label='Analytical')
ax.plot(Y_eu[:, 0] * 1e3, Y_eu[:, 1] * 1e3, '--', color='orange', lw=1.5, label=f'Euler  (h={h_fine})')
ax.plot(Y_rk[:, 0] * 1e3, Y_rk[:, 1] * 1e3,  ':',  color='red', lw=2.0, label=f'RK4    (h={h_fine})')

ax.set_xlabel('Charge q (mC)')
ax.set_ylabel('Current i (mA)')
ax.set_title('Figure 3 — Phase Portrait: i vs q\n(finest step size h = 0.0001 s)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()
plt.close()


### Discussion of Visual Differences

## Section 5: Stability Analysis

In [ ]:
h_unstable = 0.0025    # expected to destabilise Euler
h_stable   = 0.001     # RK4 still stable, Euler marginal

t_span_short = (0.0, 0.03)   # short window to see blow-up clearly

t_eu_u, Y_eu_u = euler_forward(rlc_rhs, t_span_short, y_init, h_unstable)
t_rk_u, Y_rk_u = rk4(rlc_rhs,          t_span_short, y_init, h_unstable)
t_ref_s = np.linspace(*t_span_short, 3000)
q_ref_s, i_ref_s = analytical_solution(t_ref_s)

fig, axes = plt.subplots(1, 2, figsize=(5.9, 2.2))
fig.suptitle(f'Figure 4 — Stability Demonstration (h = {h_unstable} s)', fontweight='bold')
scale = 1e3
q_ref_scaled = q_ref_s * scale #Charge

axes[0].plot(t_ref_s, q_ref_scaled, 'k-', lw=2, label='Analytical')
axes[0].plot(t_eu_u, Y_eu_u[:, 0] * scale, '--', lw=1.5, label=f'Euler (h={h_unstable})')
axes[0].plot(t_rk_u, Y_rk_u[:, 0] * scale, ':', lw=2.0, label=f'RK4 (h={h_unstable})')

axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Charge q (mC)')
axes[0].set_title('Charge q — Euler Instability')

axes[0].legend()
axes[0].grid(True, alpha=0.35)

q_lim = np.max(np.abs(q_ref_scaled)) * 4
axes[0].set_ylim(-q_lim, q_lim)

i_ref_scaled = i_ref_s * scale #Current

axes[1].plot(t_ref_s, i_ref_scaled, 'k-', lw=2, label='Analytical')
axes[1].plot(t_eu_u, Y_eu_u[:, 1] * scale, '--', lw=1.5, label=f'Euler (h={h_unstable})')
axes[1].plot(t_rk_u, Y_rk_u[:, 1] * scale, ':', lw=2.0, label=f'RK4 (h={h_unstable})')

axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Current i (mA)')
axes[1].set_title('Current i — Euler Instability')

axes[1].legend()
axes[1].grid(True, alpha=0.35)

i_lim = np.max(np.abs(i_ref_scaled)) * 4
axes[1].set_ylim(-i_lim, i_lim)

plt.show()
plt.close()


In [ ]:
h_values = np.array([8e-5, 5e-5, 3e-5, 1e-5, 5e-6, 2e-6])

t_eval = 0.05 
q_exact, _ = analytical_solution(t_eval) # Evaluate error at t_final = 0.05 s (after ~8 cycles)
euler_errors, rk4_errors = [], []
for h in h_values:
    t_e, Y_e = euler_forward(rlc_rhs, (0, t_eval), y_init, h) # Euler
    euler_errors.append(abs(Y_e[-1, 0] - q_exact))

    t_r, Y_r = rk4(rlc_rhs, (0, t_eval), y_init, h) # RK4
    rk4_errors.append(abs(Y_r[-1, 0] - q_exact))

euler_errors = np.array(euler_errors)
rk4_errors   = np.array(rk4_errors)

def log_slope(h_arr, err_arr):
    
    coeffs = np.polyfit(np.log10(h_arr), np.log10(err_arr), 1)
    return coeffs[0]

slope_euler = log_slope(h_values, euler_errors)
slope_rk4   = log_slope(h_values, rk4_errors)

fig, ax = plt.subplots(figsize=(4.0, 2.7))
ax.loglog(h_values, euler_errors, 'o--', color='red', lw=1.8, label=f"Euler  (slope ≈ {slope_euler:.2f}, expected 1)")
ax.loglog(h_values, rk4_errors,   's-',  color='blue', lw=1.8, label=f"RK4    (slope ≈ {slope_rk4:.2f}, expected 4)")

h_ref = h_values[[0, -1]] # Reference lines
ax.loglog(h_ref, 5e-7 * (h_ref / h_ref[0])**1, 'k:',  lw=1.2, label='O(h¹) reference')
ax.loglog(h_ref, 1e-12 * (h_ref / h_ref[0])**4, 'k--', lw=1.2, label='O(h⁴) reference')

ax.set_xlabel('Step size h (s)')
ax.set_ylabel('Absolute error in q(t=0.05 s)  (C)')
ax.set_title('Figure 5 — Convergence: Absolute Error vs Step Size', fontweight='bold')
ax.legend()
ax.grid(True, which='both', alpha=0.35)
plt.show()
plt.close()

print(f"Measured convergence order — Euler: {slope_euler:.2f} | RK4: {slope_rk4:.2f}")


In [ ]:
def circuit_energy(q, i):
    
    return 0.5 * q**2 / C + 0.5 * L * i**2

t_rk_f,  Y_rk_f  = rk4_results[0.0001] # Compare h_fine (stable) and h_unstable for Euler, h_fine for RK4
t_eu_f,  Y_eu_f  = euler_results[0.0001]
t_eu_m,  Y_eu_m  = euler_results[0.001]   # marginal Euler

E_ref  = circuit_energy(q_ref, i_ref)
E_rk_f = circuit_energy(Y_rk_f[:, 0], Y_rk_f[:, 1])
E_eu_f = circuit_energy(Y_eu_f[:, 0], Y_eu_f[:, 1])
E_eu_m = circuit_energy(Y_eu_m[:, 0], Y_eu_m[:, 1])

fig, ax = plt.subplots(figsize=(4.5, 2.2))
ax.semilogy(t_ref,  E_ref  * 1e6, 'k-',  lw=2,   label='Analytical')
ax.semilogy(t_rk_f, E_rk_f * 1e6, ':',   color='red', lw=2.5, label='RK4  (h=0.0001)')
ax.semilogy(t_eu_f, E_eu_f * 1e6, '--',  color='blue',  lw=1.5, label='Euler (h=0.0001)')
ax.semilogy(t_eu_m, E_eu_m * 1e6, '-.',  color='green',  lw=1.5, label='Euler (h=0.001)')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Total circuit energy (μJ)')
ax.set_title('Figure 6 — Circuit Energy vs Time\n(energy must decay for a damped system)', fontweight='bold')
ax.legend()
ax.grid(True, which='both', alpha=0.35)
plt.show()


### 5.1 Stability Limits

## Section 6: Conclusions

# COE 311K Final Project — Part 2: Stiff ODEs and Implicit Methods
## RC Circuit with Fast Transient: Spacecraft Signal Conditioning

## Section 1: Introduction & System Selection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 80
})

R       = 100.0          # Resistance     [Ω]     — NASA-HDBK-4001
C_cap   = 1e-9           # Capacitance    [F]     — 1 nF filter cap
lam     = 1.0 / (R * C_cap)  # λ = 1/RC   [s⁻¹] = 1e7
A_src   = 5.0            # Source amplitude [V]
omega_s = 1.0            # Source frequency [rad/s]  (slow signal)

V0      = 0.0            # Initial capacitor voltage [V]
t_span  = (0.0, 5.0)     # 5 seconds — captures many slow oscillations

print(f"Decay rate  λ       = {lam:.2e} s⁻¹")
print(f"RC time constant τ  = {1/lam:.2e} s  ({1/lam*1e6:.3f} µs)")
print(f"Input period T_src  = {2*np.pi/omega_s:.3f} s")
print(f"Stiffness ratio     ≈ {lam / omega_s:.2e}  (λ / ω_source)")


## Section 2: Demonstration of Stiffness

In [ ]:
def rc_rhs(t, V):
    
    return -lam * V + lam * A_src * np.sin(omega_s * t)

def euler_forward_scalar(rhs, t_span, y0, h):
    
    t0, tf = t_span
    t = np.arange(t0, tf + h * 0.5, h)
    y = np.zeros(len(t))
    y[0] = y0
    for n in range(len(t) - 1):
        y[n + 1] = y[n] + h * rhs(t[n], y[n])
    return t, y

def analytical_V(t):
    
    A = A_src * lam
    a_p =  A * lam   / (lam**2 + omega_s**2)   # sin coefficient
    b_p = -A * omega_s / (lam**2 + omega_s**2)  # cos coefficient
    C0  = V0 - (a_p * np.sin(0) + b_p * np.cos(0))
    return C0 * np.exp(-lam * t) + a_p * np.sin(omega_s * t) + b_p * np.cos(omega_s * t)

t_short = (0.0, 0.02)
h_eu_stable   = 1e-7       # just inside stability boundary
h_eu_unstable = 3e-7       # just outside — should blow up

t_ref_s = np.linspace(*t_short, 10000)
V_ref_s = analytical_V(t_ref_s)

t_eu_s, V_eu_s   = euler_forward_scalar(rc_rhs, t_short, V0, h_eu_stable)
t_eu_u, V_eu_u   = euler_forward_scalar(rc_rhs, t_short, V0, h_eu_unstable)

fig, axes = plt.subplots(1, 2, figsize=(5.9, 2.2))
fig.suptitle('Figure 1 — Euler Forward Instability on the Stiff RC Circuit', fontweight='bold', y = 1.1)

ax = axes[0]
te = t_eu_s
Ve = V_eu_s
h_val = h_eu_stable
stable_flag = True

ax.plot(t_ref_s, V_ref_s, 'k-', lw=2, label='Analytical', zorder=5)
status = 'stable (barely)' if stable_flag else 'UNSTABLE'
ax.plot(te, np.clip(Ve, -20, 20), '--',
        color='red', lw=1.5,
        label=f'Euler Forward h={h_val:.0e} s  [{status}]')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Voltage V (V)')
ax.set_title(f'Euler Forward  h = {h_val:.0e} s')
ax.set_ylim(-15, 15)
ax.legend()
ax.grid(True, alpha=0.35)

ax = axes[1]
te = t_eu_u
Ve = V_eu_u
h_val = h_eu_unstable
stable_flag = False

ax.plot(t_ref_s, V_ref_s, 'k-', lw=2, label='Analytical', zorder=5)
status = 'stable (barely)' if stable_flag else 'UNSTABLE'
ax.plot(te, np.clip(Ve, -20, 20), '--',
        color='red', lw=1.5,
        label=f'Euler Forward h={h_val:.0e} s  [{status}]')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Voltage V (V)')
ax.set_title(f'Euler Forward  h = {h_val:.0e} s')
ax.set_ylim(-15, 15)
ax.legend()
ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()
plt.close()

h_stable_euler = 2.0 / lam
T_sim = t_span[1] - t_span[0]
N_required = T_sim / h_stable_euler

print("\n── Stiffness Analysis ──────────────────────────────────────────")
print(f"Eigenvalue λ                     = {lam:.2e} s⁻¹")
print(f"Forcing frequency ω              = {omega_s:.1f} rad/s")
print(f"Stiffness ratio                  = {lam/omega_s:.2e}")
print(f"Max stable h (Euler Forward)     = {h_stable_euler:.2e} s")
print(f"Steps needed for 5 s simulation  = {N_required:.2e}  ← impractical")
print(f"Max stable h (Euler Backward)    = UNLIMITED  (A-stable method)")


## Section 3: Mathematical Setup

In [ ]:
def residual_g(V_next, V_curr, t_next, h):
    
    return V_next - V_curr - h * rc_rhs(t_next, V_next)

def residual_g_prime(h):
    
    return 1.0 + h * lam

def newton_raphson(V_curr, t_next, h, tol=1e-8, max_iter=50):
    
    V_next = V_curr   # initial guess: explicit Euler predictor (V_n)
    gp = residual_g_prime(h)   # constant Jacobian — compute once

    for k in range(max_iter):
        g_val = residual_g(V_next, V_curr, t_next, h)

        if abs(g_val) < tol:   # convergence check
            return V_next, k + 1

        V_next = V_next - g_val / gp   # Newton-Raphson update

    return V_next, max_iter   # return best estimate if max_iter reached

def euler_backward(rhs, t_span, y0, h, tol=1e-8):
    
    t0, tf = t_span
    t = np.arange(t0, tf + h * 0.5, h)
    y = np.zeros(len(t))
    nr_iters = np.zeros(len(t) - 1, dtype=int)
    y[0] = y0

    for n in range(len(t) - 1):
        y[n + 1], nr_iters[n] = newton_raphson(y[n], t[n + 1], h, tol)

    return t, y, nr_iters


In [ ]:
h_list = [0.1, 0.01, 0.001]       # much larger than Euler Forward's limit!

eb_results = {}
for h in h_list:
    t_eb, V_eb, nr_it = euler_backward(rc_rhs, t_span, V0, h)
    eb_results[h] = (t_eb, V_eb, nr_it)

t_ref = np.linspace(*t_span, 50000)
V_ref = analytical_V(t_ref)

print("✓ Euler Backward solutions computed.")
for h in h_list:
    t_e, V_e, nr_i = eb_results[h]
    print(f"  h = {h:.3f} s  |  steps = {len(t_e)-1:6d}  |  avg NR iter/step = {nr_i.mean():.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 2.2))
ax.plot(t_ref, V_ref, 'k-', lw=2.5, label='Analytical (exact)', zorder=5)

h = h_list[0]
t_e, V_e, _ = eb_results[h]
ax.plot(t_e, V_e, '--', color='orange', lw=1.6,
        label=f'Euler Backward  h = {h} s')

h = h_list[1]
t_e, V_e, _ = eb_results[h]
ax.plot(t_e, V_e, '--', color='red', lw=1.6,
        label=f'Euler Backward  h = {h} s')

h = h_list[2]
t_e, V_e, _ = eb_results[h]
ax.plot(t_e, V_e, '--', color='blue', lw=1.6,
        label=f'Euler Backward  h = {h} s')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Capacitor Voltage V (V)')
ax.set_title('Figure 2 — Euler Backward: Solution at Multiple Step Sizes\n'
             'RC Circuit (λ = 10⁷ s⁻¹, V_source = 5 sin(t) V)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.35)
plt.show()
plt.close()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(6.3, 1.8))
fig.suptitle('Figure 4 — Newton-Raphson Iterations per Time Step', fontweight='bold')

plt.subplots_adjust(top=0.80, wspace=0.3)

h0 = h_list[0]
ax0 = axes[0]

t_e0, V_e0, nr_i0 = eb_results[h0]
ax0.bar(t_e0[1:], nr_i0, width=h0 * 0.8, color='orange', alpha=0.75)
ax0.set_xlabel('Time (s)')
ax0.set_title(f'h = {h0} s\n(avg = {nr_i0.mean():.2f} iter/step)')
ax0.set_ylim(0, nr_i0.max() + 2)
ax0.grid(True, alpha=0.35)
ax0.set_ylabel('NR iterations')

h1 = h_list[1]
ax1 = axes[1]

t_e1, V_e1, nr_i1 = eb_results[h1]
ax1.bar(t_e1[1:], nr_i1, width=h1 * 0.8, color='red', alpha=0.75)
ax1.set_xlabel('Time (s)')
ax1.set_title(f'h = {h1} s\n(avg = {nr_i1.mean():.2f} iter/step)')
ax1.set_ylim(0, nr_i1.max() + 2)
ax1.grid(True, alpha=0.35)

h2 = h_list[2]
ax2 = axes[2]

t_e2, V_e2, nr_i2 = eb_results[h2]
ax2.bar(t_e2[1:], nr_i2, width=h2 * 0.8, color='blue', alpha=0.75)
ax2.set_xlabel('Time (s)')
ax2.set_title(f'h = {h2} s\n(avg = {nr_i2.mean():.2f} iter/step)')
ax2.set_ylim(0, nr_i2.max() + 2)
ax2.grid(True, alpha=0.35)

plt.show()
plt.close()

print("Note: Since the ODE is linear, g′ is constant. One Newton update finds the exact solution,")
print("      so NR reports 2 iterations (initial residual check + update + convergence check).")
print("      For nonlinear ODEs (e.g., Option B–E), multiple NR updates per step would be needed.")


In [ ]:
h_range   = np.array([0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.001])
t_eval    = 4.0   # evaluate error at t = 4 s (well into quasi-static regime)
V_exact_4 = float(analytical_V(t_eval))

eb_errors = []
for h in h_range:
    t_e, V_e, _ = euler_backward(rc_rhs, (0, t_eval), V0, h)
    eb_errors.append(abs(V_e[-1] - V_exact_4))

eb_errors = np.array(eb_errors)

slope_eb = np.polyfit(np.log10(h_range), np.log10(eb_errors + 1e-20), 1)[0]

fig, ax = plt.subplots(figsize=(4.0, 2.7))
ax.loglog(h_range, eb_errors, 'o-', color='#4a90d9', lw=2,
          label=f'Euler Backward (slope ≈ {slope_eb:.2f}, expected 1)')

h_r = h_range[[0, -1]]
ax.loglog(h_r, 1.0 * (h_r / h_r[0])**1, 'k--', lw=1.2, label='O(h¹) reference')

ax.set_xlabel('Step size h (s)')
ax.set_ylabel('Absolute error at t = 4 s  (V)')
ax.set_title('Figure 5 — Convergence Rate: Error vs Step Size (Euler Backward)', fontweight='bold')
ax.legend()
ax.grid(True, which='both', alpha=0.35)
plt.show()
plt.close()

print(f"Measured convergence order (Euler Backward): {slope_eb:.2f}  (theoretical: 1.0)")


### Discussion of Results

In [ ]:
T_sim = t_span[1] - t_span[0]   # 5 s
h_ef_max = 2.0 / lam            # max stable Euler Forward step
N_ef = int(np.ceil(T_sim / h_ef_max))

print("Euler Forward (stable):")
print(f"  Step size: {h_ef_max:.2e} s")
print(f"  Number of steps: {N_ef:,}")
print(f"  Newton iterations per step: 1 (explicit)\n")

print("Euler Backward results:")

for h in h_list:
    t_e, V_e, nr_i = eb_results[h]
    N_steps = len(t_e) - 1
    avg_nr = nr_i.mean()

    print(f"  h = {h} s")
    print(f"    steps: {N_steps:,}")
    print(f"    avg Newton iterations/step: {avg_nr:.1f}\n")

print("Comparison (h = 0.1 s):")
N_eb = len(eb_results[0.1][0]) - 1

print(f"  Euler Forward steps: {N_ef:,}")
print(f"  Euler Backward steps: {N_eb:,}")
print(f"  Speedup: {N_ef / N_eb:,.0f}× fewer steps")


In [ ]:
name0 = 'Euler Forward\n(stable, h=2e-7 s)'
name1 = 'EB  h=0.1 s'
name2 = 'EB  h=0.01 s'
name3 = 'EB  h=0.001 s'

n0 = N_ef
n1 = len(eb_results[0.1][0]) - 1
n2 = len(eb_results[0.01][0]) - 1
n3 = len(eb_results[0.001][0]) - 1

c0 = '#e07b39'
c1 = '#4a90d9'
c2 = '#4a90d9'
c3 = '#4a90d9'

fig, ax = plt.subplots(figsize=(4.0, 2.2))

bars = ax.bar([name0, name1, name2, name3],
              [n0, n1, n2, n3],
              color=[c0, c1, c2, c3],
              alpha=0.85,
              edgecolor='k',
              linewidth=0.8)

ax.set_yscale('log')
ax.set_ylabel('Number of time steps (log scale)')
ax.set_title('Figure 6 — Computational Cost Comparison\nEuler Forward (explicit) vs Euler Backward (implicit)',
             fontweight='bold')

ax.grid(True, which='both', axis='y', alpha=0.35)

b0, b1, b2, b3 = bars[0], bars[1], bars[2], bars[3]

ax.text(b0.get_x() + b0.get_width()/2,
        n0 * 1.2,
        f'{n0:,}', ha='center', va='bottom',
        fontsize=10, fontweight='bold')

ax.text(b1.get_x() + b1.get_width()/2,
        n1 * 1.2,
        f'{n1:,}', ha='center', va='bottom',
        fontsize=10, fontweight='bold')

ax.text(b2.get_x() + b2.get_width()/2,
        n2 * 1.2,
        f'{n2:,}', ha='center', va='bottom',
        fontsize=10, fontweight='bold')

ax.text(b3.get_x() + b3.get_width()/2,
        n3 * 1.2,
        f'{n3:,}', ha='center', va='bottom',
        fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
plt.close()


In [ ]:
print("──────────────────────────────────────────────")
print("Accuracy vs Computational Cost Comparison")
print("──────────────────────────────────────────────\n")

ef_steps = N_ef
ef_f_evals = ef_steps * 1  # explicit method

print("Euler Forward (stable):")
print(f"  Step size: ~{h_ef_max:.2e} s")
print(f"  Total steps: {ef_steps:,}")
print(f"  Function evaluations: {ef_f_evals:,}\n")

print("Euler Backward:")

for h in eb_results:
    t_e, V_e, nr_i = eb_results[h]

    steps = len(t_e) - 1
    avg_nr = nr_i.mean()
    f_evals = steps * avg_nr

    error = np.abs(V_e - analytical_V(t_e)).mean()

    print(f"h = {h}")
    print(f"  steps: {steps:,}")
    print(f"  avg NR iterations/step: {avg_nr:.2f}")
    print(f"  function evals: ~{f_evals:,.0f}")
    print(f"  mean error: {error:.3e}\n")
    
print("Key comparison:")
print(f"  Euler Forward requires ~{ef_steps:,} steps")
print(f"  Euler Backward (h=0.1) requires ~{len(eb_results[0.1][0])-1:,} steps")
print(f"  → Backward method achieves stability with far fewer steps for stiff problems")


### Performance Summary

## Section 7: Conclusions

# COE 311K Final Project — Part 3: Damping and Adaptive Step Sizes
## Section 1: Review of Part 2

In [ ]:
import matplotlib.gridspec as gridspec

R_val = 100.0          # Resistance     [Ω]
C_cap = 1e-9           # Capacitance    [F]  (1 nF)
lam = 1.0 / (R_val * C_cap)   # λ = 1/RC  [s⁻¹] = 1e7
A_src = 5.0            # Source amplitude [V]
omega_s = 1.0          # Source frequency [rad/s]
V0 = 0.0               # Initial capacitor voltage [V]
t_span = (0.0, 5.0)    # Simulation window [s]

print("── Part 2 System Recap ─────────────────────────────────")
print(f"  R = {R_val} Ω,  C = {C_cap:.0e} F")
print(f"  λ = 1/RC = {lam:.2e} s⁻¹  (stiffness eigenvalue)")
print(f"  Stiffness ratio |λ|/ω = {lam/omega_s:.2e}")
print(f"  Max stable h (Euler Forward) = {2/lam:.2e} s")
print(f"  Euler Forward steps for 5 s  = {5/(2/lam):.2e}  ← impractical")
print(f"  Euler Backward: A-stable, h limited only by accuracy")


## Section 2: Newton-Raphson Damping Implementation

In [ ]:
def basic_newton_raphson(V_curr, t_next, h, tol=1e-8, max_iter=50):
    
    V_guess = V_curr
    g_prime = residual_g_prime(h)   # constant for this linear ODE
 
    for k in range(max_iter):
        g = residual_g(V_guess, V_curr, t_next, h)
 
        if abs(g) < tol:
            return V_guess, k, True
 
        V_guess = V_guess - g / g_prime
 
    return V_guess, max_iter, False


In [ ]:
def damped_newton_raphson(V_curr, t_next, h,
                          tol=1e-8, max_iter=50, alpha_min=1e-4,
                          max_bt=10):
    V_guess        = V_curr
    g_prime        = residual_g_prime(h)
    total_bt_steps = 0
    alpha_used     = 1.0
    damping_needed = False

    for k in range(max_iter):
        g = residual_g(V_guess, V_curr, t_next, h)

        if abs(g) < tol:
            return V_guess, k, True, total_bt_steps, alpha_used, damping_needed

        delta = -g / g_prime
        g_current = abs(g)
        alpha = 1.0

        for bt in range(max_bt):
            V_trial = V_guess + alpha * delta
            g_trial = abs(residual_g(V_trial, V_curr, t_next, h))

            if g_trial < g_current:
                break

            alpha = alpha / 2.0
            total_bt_steps += 1

            if alpha < alpha_min:
                return V_guess, k, False, total_bt_steps, alpha, damping_needed

        if alpha < 1.0:
            damping_needed = True

        alpha_used = alpha
        V_guess    = V_guess + alpha * delta

    return V_guess, max_iter, False, total_bt_steps, alpha_used, damping_needed


In [ ]:
print("─" * 60)
print("Section 2: Newton-Raphson Damping Demonstration")
print("─" * 60)
 
h_large  = 2.0       # very large step — challenging for basic NR
t_next_demo = 0.0 + h_large   # from t=0 to t=h_large
V_curr_demo = 0.0              # initial voltage
 
try:
    V_und, iters_und, conv_und = basic_newton_raphson(
        V_curr_demo, t_next_demo, h_large, tol=1e-8, max_iter=50
    )
    residual_und = abs(residual_g(V_und, V_curr_demo, t_next_demo, h_large))
    print(f"\nUndamped NR  (h = {h_large} s):")
    print(f"  Converged: {conv_und}")
    print(f"  Iterations: {iters_und}")
    print(f"  Final residual |g|: {residual_und:.3e}")
    print(f"  V_next = {V_und:.6f} V")
except Exception as e:
    print(f"  Undamped NR raised exception: {e}")

V_dmp, iters_dmp, conv_dmp, bt_steps, alpha_f, damp_needed = \
    damped_newton_raphson(
        V_curr_demo, t_next_demo, h_large,
        tol=1e-8, max_iter=50, alpha_min=1e-6
    )
residual_dmp = abs(residual_g(V_dmp, V_curr_demo, t_next_demo, h_large))
 
print(f"\nDamped NR    (h = {h_large} s):")
print(f"  Converged: {conv_dmp}")
print(f"  Outer NR iterations: {iters_dmp}")
print(f"  Backtracking steps:  {bt_steps}")
print(f"  Final alpha:         {alpha_f:.4f}")
print(f"  Damping needed:      {damp_needed}")
print(f"  Final residual |g|:  {residual_dmp:.3e}")
print(f"  V_next = {V_dmp:.6f} V")
 
V_exact_demo = analytical_V(np.array([t_next_demo]))[0]
print(f"\n  Analytical V(t={t_next_demo:.1f}) = {V_exact_demo:.6f} V")
print(f"  Error (damped NR):   {abs(V_dmp - V_exact_demo):.3e} V")


## Section 3: Adaptive Step Size Control

In [ ]:
def implicit_euler_step(y_n, t_n, t_next, h, tol_nr=1e-8):
    y_next, iters, converged, _, _, _ = damped_newton_raphson(y_n, t_next, h, tol=tol_nr)
    return y_next, iters


In [ ]:
def estimate_error_step_doubling(y_n, t_n, h):
    y_full, iters_full = implicit_euler_step(y_n, t_n, t_n + h, h)
 
    y_half_1, iters_1 = implicit_euler_step(y_n,      t_n,       t_n + h/2, h/2)
 
    y_half, iters_2   = implicit_euler_step(y_half_1, t_n + h/2, t_n + h,   h/2)
 
    error = abs(y_half - y_full)
 
    total_iters = iters_full + iters_1 + iters_2
 
    return y_full, y_half, error, total_iters


In [ ]:
def adjust_step_size(h_current, error, tol, safety=0.9, h_min=1e-6, h_max=1.0):
    if error < 1e-14:
        h_new = min(2.0 * h_current, h_max)
    else:
        h_new = h_current * np.sqrt(tol / error)
        h_new = safety * h_new
 
    h_new = max(h_min, min(h_max, h_new))
 
    h_new = max(0.2 * h_current, min(5.0 * h_current, h_new))
 
    return h_new


In [ ]:
def adaptive_step(y_n, t_n, h, tol, h_min=1e-6, h_max=1.0, safety=0.9):
    y_full, y_half, error, iters = estimate_error_step_doubling(y_n, t_n, h)
 
    h_new = adjust_step_size(h, error, tol, safety=safety, h_min=h_min, h_max=h_max)
 
    if error < tol:
        return y_half, h, h_new, True, iters, error
    else:
        return y_n, h, h_new, False, iters, error


In [ ]:
def adaptive_implicit_euler(y0, t0, t_final, h0, tol, h_min=1e-6, h_max=1.0, safety=0.9, max_steps=100000):
    t = [t0]
    y = [y0]
    h_history   = []
    error_history = []
 
    stats = {
        'accepted_steps'  : 0,
        'rejected_steps'  : 0,
        'nr_iterations'   : [],   # per accepted step
        'function_evals'  : 0,
        'damping_used'    : 0,
    }
 
    h = h0
    n_steps = 0
 
    while t[-1] < t_final and n_steps < max_steps:
        t_n = t[-1]
        y_n = y[-1]
 
        h = min(h, t_final - t_n)
        if h < h_min:
            h = h_min
 
        y_next, h_used, h_new, accepted, iters, error = adaptive_step(
            y_n, t_n, h, tol, h_min=h_min, h_max=h_max, safety=safety
        )
 
        stats['function_evals'] += iters
 
        if accepted:
            t.append(t_n + h_used)
            y.append(y_next)
            h_history.append(h_used)
            error_history.append(error)
            stats['accepted_steps'] += 1
            stats['nr_iterations'].append(iters)
        else:
            stats['rejected_steps'] += 1
 
        h = h_new
        n_steps += 1
 
    stats['error_history'] = np.array(error_history)
    return np.array(t), np.array(y), np.array(h_history), stats


In [ ]:
t0, t_final = t_span
tol_adapt   = 1e-4      # target local error tolerance
h0_adapt    = 0.01      # initial step size
 
t_adapt, y_adapt, h_hist, stats_adapt = adaptive_implicit_euler(
    y0=V0, t0=t0, t_final=t_final, h0=h0_adapt, tol=tol_adapt,
    h_min=1e-6, h_max=1.0, safety=0.9
)

t_ref  = np.linspace(t0, t_final, 5000)
V_ref  = analytical_V(t_ref)
 
print(f"\n── Adaptive Solver Results ─────────────────────────────")
print(f"  Accepted steps:       {stats_adapt['accepted_steps']}")
print(f"  Rejected steps:       {stats_adapt['rejected_steps']}")
print(f"  Total step attempts:  {stats_adapt['accepted_steps'] + stats_adapt['rejected_steps']}")
print(f"  Total NR iterations:  {stats_adapt['function_evals']}")
print(f"  Time points stored:   {len(t_adapt)}")
print(f"  Min h used:           {h_hist.min():.4e} s")
print(f"  Max h used:           {h_hist.max():.4e} s")
V_exact_at_t = analytical_V(t_adapt)
max_err = np.max(np.abs(y_adapt - V_exact_at_t))
mean_err = np.mean(np.abs(y_adapt - V_exact_at_t))
print(f"  Max absolute error:   {max_err:.4e} V")
print(f"  Mean absolute error:  {mean_err:.4e} V")


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 2.2))
 
ax.plot(t_ref, V_ref, 'k-', lw=2, label='Analytical', zorder=5)
ax.plot(t_adapt, y_adapt, 'b-o', lw=1.5, markersize=3, label=f'Adaptive Euler Backward  ({stats_adapt["accepted_steps"]} points, tol={tol_adapt:.0e})', zorder=4)
 
ax.set_xlabel('Time t (s)')
ax.set_ylabel('Voltage V (V)')
ax.set_title('Figure 3 — Adaptive Implicit Euler Solution\n' '(circles = accepted time points; spacing reflects step-size adaptation)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()
plt.close()


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 1.8))

ax.plot(t_adapt[1:], h_hist, 'r-o', lw=1.5, markersize=3)

ax.axhline(tol_adapt, color='gray', linestyle='--', lw=1.2, label=f'tolerance = {tol_adapt:.0e}')

ax.set_xlabel('Time t (s)')
ax.set_ylabel('Step size h (s)')
ax.set_title('Figure 4 — Adaptive Step Size h(t) vs. Time\n' '(h shrinks in difficult regions, grows when solution is smooth)', fontweight='bold')

ax.legend()
ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()
plt.close()


In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 1.8))
error_history = stats_adapt['error_history']  # Get it from stats dictionary
ax.plot(t_adapt[1:len(error_history) + 1], error_history, 'g-o',
        lw=1.5, markersize=3, label='Step-doubling error estimate')
ax.axhline(tol_adapt, color='k', linestyle='--', lw=1.5,
           label=f'Tolerance = {tol_adapt:.0e}')
ax.set_xlabel('Time t (s)')
ax.set_ylabel('Error estimate [V]')
ax.set_title('Figure 5 — Error Estimate vs. Time\n'
             '(error stays below tolerance at all accepted steps)',
             fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()
plt.close()


## Section 4: Performance Analysis — Fixed vs. Adaptive Step Size

In [ ]:
def euler_backward_fixed(y0, t0, t_final, h, tol_nr=1e-8):
    t = np.arange(t0, t_final + h * 0.5, h)
    y = np.zeros(len(t))
    y[0] = y0
    n_iters = 0
 
    for n in range(len(t) - 1):
        y_next, iters, _, _, _, _ = damped_newton_raphson(
            y[n], t[n + 1], h, tol=tol_nr
        )
        y[n + 1]  = y_next
        n_iters  += iters
 
    return t, y, n_iters

h_fixed_vals = [0.1, 0.01, 0.001]
fixed_results = {}
for h_f in h_fixed_vals:
    t_f, y_f, ni_f = euler_backward_fixed(V0, t0, t_final, h_f)
    V_exact_f      = analytical_V(t_f)
    err_f          = np.max(np.abs(y_f - V_exact_f))
    fixed_results[h_f] = {
        't': t_f, 'y': y_f,
        'steps': len(t_f) - 1,
        'nr_iters': ni_f,
        'max_error': err_f
    }

V_exact_adapt = analytical_V(t_adapt)
max_err_adapt = np.max(np.abs(y_adapt - V_exact_adapt))

print("\n--- Method Comparison ---")

for h_f, res in fixed_results.items():
    print(f"Fixed h={h_f:.3f}: steps={res['steps']}, "
          f"NR iters={res['nr_iters']}, max error={res['max_error']:.4e}")

print(f"\nAdaptive (tol=1e-4): steps={stats_adapt['accepted_steps']}, "
      f"NR iters={stats_adapt['function_evals']}, max error={max_err_adapt:.4e}")

total_attempts = stats_adapt['accepted_steps'] + stats_adapt['rejected_steps']
acceptance_pct = 100 * stats_adapt['accepted_steps'] / total_attempts

print(f"Adaptive acceptance rate: {acceptance_pct:.1f}% "
      f"({stats_adapt['accepted_steps']}/{total_attempts})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5.9, 2.2))
fig.suptitle('Figure 6 — Fixed vs. Adaptive Step Size Performance Comparison', fontweight='bold', y=1)
for h_f in fixed_results:
    res = fixed_results[h_f]
    axes[0].scatter(res['max_error'], res['nr_iters'], s=80, color='blue')
    axes[0].text(res['max_error'], res['nr_iters'],
                 f'h={h_f}', fontsize=9, color='blue',
                 ha='left', va='bottom')

axes[0].scatter(max_err_adapt,
                stats_adapt['function_evals'],
                s=150, color='red', marker='*')
axes[0].text(max_err_adapt,
             stats_adapt['function_evals'],
             'Adaptive',
             fontsize=9, color='red',
             ha='left', va='bottom')

axes[0].set_xlabel('Max absolute error (V)')
axes[0].set_ylabel('Total NR iterations')
axes[0].set_title('Cost vs Accuracy\n(lower is better)')
axes[0].grid(True, alpha=0.35)

labels = []
steps = []
colors = []

for h_f in h_fixed_vals:
    labels.append(f'Fixed h={h_f}')
    steps.append(fixed_results[h_f]['steps'])
    colors.append('#4a90d9')

labels.append('Adaptive')
steps.append(stats_adapt['accepted_steps'])
colors.append('#e07b39')

bars = axes[1].bar(labels, steps, color=colors, alpha=0.85,
                   edgecolor='k', linewidth=0.8)

axes[1].set_ylabel('Number of steps')
axes[1].set_title('Steps Required', pad = 12)
axes[1].grid(True, axis='y', alpha=0.35)

for i in range(len(bars)):
    bar = bars[i]
    val = steps[i]
    axes[1].text(bar.get_x() + bar.get_width()/2, val * 1.05, f'{val:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()
plt.close()


In [ ]:
tol_show  = 1e-3
h0_show   = 0.05
 
t_show, y_show, h_hist_show, stats_show = adaptive_implicit_euler(
    V0, t0, t_final, h0=h0_show, tol=tol_show,
    h_min=1e-6, h_max=1.0, safety=0.9
)
 
def adaptive_with_log(y0, t0, t_final, h0, tol, h_min=1e-6, h_max=1.0, safety=0.9):
    
    t_log, y_log   = [t0], [y0]
    attempts_t     = []
    attempts_acc   = []   # True = accepted, False = rejected
    h = h0
 
    while t_log[-1] < t_final:
        t_n = t_log[-1]
        y_n = y_log[-1]
        h   = min(h, t_final - t_n)
        if h < h_min:
            h = h_min
 
        y_next, h_used, h_new, accepted, iters, error = adaptive_step(
            y_n, t_n, h, tol, h_min=h_min, h_max=h_max, safety=safety
        )
 
        attempts_t.append(t_n + h)
        attempts_acc.append(accepted)
 
        if accepted:
            t_log.append(t_n + h_used)
            y_log.append(y_next)
 
        h = h_new
        if len(t_log) > 50000:
            break
 
    return np.array(t_log), np.array(y_log), np.array(attempts_t), np.array(attempts_acc)
 
t_l, y_l, att_t, att_acc = adaptive_with_log(
    V0, t0, min(t_final, 2.0), h0_show, tol_show
)
 
fig, ax = plt.subplots(figsize=(5.4, 2.2))
ax.plot(t_ref[t_ref <= 2.0], analytical_V(t_ref[t_ref <= 2.0]),
        'k-', lw=2, label='Analytical', zorder=5)

acc_mask = att_acc == True # Accepted step endpoints
rej_mask = att_acc == False
ax.scatter(att_t[acc_mask], analytical_V(att_t[acc_mask]), color='#27ae60', s=40, zorder=6, label='Accepted step endpoint')
ax.scatter(att_t[rej_mask], analytical_V(att_t[rej_mask]), color='red', s=40, marker='x', zorder=6, linewidths=2, label='Rejected step endpoint')
 
ax.set_xlabel('Time t (s)')
ax.set_ylabel('Voltage V (V)')
ax.set_title(f'Figure 7 — Accepted vs. Rejected Steps  'f'(tol={tol_show:.0e},  h₀={h0_show})\n' f'Green = accepted, Red ✕ = rejected (step retried with smaller h)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()
 
n_acc = int(acc_mask.sum())
n_rej = int(rej_mask.sum())
print(f"Accepted: {n_acc},  Rejected: {n_rej},  "f"Acceptance rate: {100*n_acc/(n_acc+n_rej):.1f}%")


In [ ]:
def euler_backward_fixed_basic(y0, t0, t_final, h, tol_nr=1e-8):
    
    t = np.arange(t0, t_final + h * 0.5, h) # Run fixed-step solver with undamped NR for comparison
    y = np.zeros(len(t))
    y[0] = y0
    iters_per_step = []
 
    for n in range(len(t) - 1):
        y_next, iters, conv = basic_newton_raphson(
            y[n], t[n + 1], h, tol=tol_nr, max_iter=50
        )
        y[n + 1] = y_next
        iters_per_step.append(iters)
 
    return t, y, np.array(iters_per_step)

h_compare = 0.01
t_basic, y_basic, iters_basic = euler_backward_fixed_basic(V0, t0, t_final, h_compare)
t_damped, y_damped, ni_damped  = euler_backward_fixed(V0, t0, t_final, h_compare)
 
err_basic  = np.abs(y_basic  - analytical_V(t_basic))
err_damped = np.abs(y_damped - analytical_V(t_damped))
 
fig, axes = plt.subplots(2, 1, figsize=(5.4, 3.1), sharex=True)
fig.suptitle(f'Figure 8 — Fixed-Step Euler Backward  (h = {h_compare} s):\n' f'Basic NR vs. Damped NR Comparison', fontweight='bold', y = 1.1)

axes[0].plot(t_ref,    analytical_V(t_ref), 'k-',  lw=2,   label='Analytical')
axes[0].plot(t_basic,  y_basic,             'r--', lw=1.5, label='Basic NR')
axes[0].plot(t_damped, y_damped,            'b:',  lw=2.0, label='Damped NR')
axes[0].set_ylabel('Voltage V (V)')
axes[0].set_title('Solution')
axes[0].legend()
axes[0].grid(True, alpha=0.35)
 
axes[1].plot(t_basic, err_basic, 'r--', lw=1.5, label=f'Basic NR (max={err_basic.max():.3e} V)')
axes[1].plot(t_damped, err_damped, 'b:', lw=2.0, label=f'Damped NR (max={err_damped.max():.3e} V)')

axes[1].set_xlabel('Time t (s)')
axes[1].set_ylabel('Absolute error |V - V_exact| (V)')
axes[1].set_title('Absolute Error vs. Time')
axes[1].legend()
axes[1].grid(True, alpha=0.35)
 
plt.tight_layout()
plt.show()
 
print(f"\nBasic NR  max error: {err_basic.max():.4e} V")
print(f"Damped NR max error: {err_damped.max():.4e} V")
print(f"For h = {h_compare} s both methods converge (linear ODE);")
print(f"damping improves robustness at much larger h (see Section 2).")


## Section 5: Robustness Testing

In [ ]:
print("Robustness Test 1: Very Large Initial Step (h₀ = 1.0 s)")
 
h0_large  = 1.0
tol_large = 1e-4
 
t_rob1, y_rob1, h_hist1, stats_rob1 = adaptive_implicit_euler(V0, t0, t_final, h0=h0_large, tol=tol_large, h_min=1e-6, h_max=1.0, safety=0.9)
 
V_exact_rob1 = analytical_V(t_rob1)
max_err_rob1  = np.max(np.abs(y_rob1 - V_exact_rob1))
 
print(f"\n  h₀ = {h0_large} s  (starts at max allowed size)")
print(f"  Tolerance: {tol_large:.0e}")
print(f"  Accepted steps:   {stats_rob1['accepted_steps']}")
print(f"  Rejected steps:   {stats_rob1['rejected_steps']}")
print(f"  Total NR iters:   {stats_rob1['function_evals']}")
print(f"  Max absolute err: {max_err_rob1:.4e} V")
print(f"  ✓ Solver handled h₀ = 1.0 s gracefully (rejected initial steps, "f"reduced h automatically)")


In [ ]:
print("Robustness Test 2: Very Tight Tolerance (tol = 1e-8)")
 
tol_tight = 1e-8
h0_tight  = 0.01
 
t_rob2, y_rob2, h_hist2, stats_rob2 = adaptive_implicit_euler(
    V0, t0, t_final, h0=h0_tight, tol=tol_tight,
    h_min=1e-6, h_max=1.0, safety=0.9
)
 
V_exact_rob2  = analytical_V(t_rob2)
max_err_rob2   = np.max(np.abs(y_rob2 - V_exact_rob2))
 
print(f"\n  Tolerance:        {tol_tight:.0e}")
print(f"  h₀ = {h0_tight} s")
print(f"  Accepted steps:   {stats_rob2['accepted_steps']}")
print(f"  Rejected steps:   {stats_rob2['rejected_steps']}")
print(f"  Total NR iters:   {stats_rob2['function_evals']}")
print(f"  Max absolute err: {max_err_rob2:.4e} V")
print(f"  ✓ Achieved tol = 1e-8 with more (smaller) steps automatically")


## Section 6: Conclusions

## Bonus Challenges

### Bonus 1: PI Controller for Step Size

In [ ]:
def adjust_step_size_pi(h_current, error_n, error_prev, tol, k_P=0.7, k_I=0.4, safety=0.9, h_min=1e-6, h_max=1.0):
    eps = 1e-14  # prevent division by zero
    p_term = (tol / (error_n   + eps)) ** k_P
    i_term = ((error_prev + eps) / (error_n + eps)) ** k_I
    h_new  = safety * h_current * p_term * i_term

    h_new = max(h_min, min(h_max, h_new)) # Enforce absolute bounds and limit change per step
    h_new = max(0.2 * h_current, min(5.0 * h_current, h_new))
    return h_new

def adaptive_implicit_euler_pi(y0, t0, t_final, h0, tol, k_P=0.7, k_I=0.4, h_min=1e-6, h_max=1.0, safety=0.9, max_steps=100_000):
    t = [t0];  y = [y0]
    h_history     = []
    error_history = []

    stats = {'accepted_steps': 0, 'rejected_steps': 0, 'function_evals': 0}

    h          = h0
    error_prev = tol   # neutral initial value (I-term starts flat)
    n_steps    = 0

    while t[-1] < t_final and n_steps < max_steps:
        t_n = t[-1];  y_n = y[-1]

        h = min(h, t_final - t_n)
        if h < h_min:
            h = h_min

        y_full, y_half, error, iters = estimate_error_step_doubling(y_n, t_n, h)
        stats['function_evals'] += iters

        h_new = adjust_step_size_pi(h, error, error_prev, tol,
                                    k_P=k_P, k_I=k_I,
                                    safety=safety, h_min=h_min, h_max=h_max)

        if error < tol:                   # ACCEPT
            t.append(t_n + h)
            y.append(y_half)              # more accurate half-step result
            h_history.append(h)
            error_history.append(error)
            stats['accepted_steps'] += 1
            error_prev = max(error, 1e-14)  # update I-term memory
        else:                             # REJECT
            stats['rejected_steps'] += 1

        h = h_new
        n_steps += 1

    stats['error_history'] = np.array(error_history)
    return np.array(t), np.array(y), np.array(h_history), stats

tol_bonus = 1e-4

print("Running P-controller solver  (tol=1e-4) …")
t_P,  y_P,  h_hist_P,  stats_P  = adaptive_implicit_euler(
    V0, t0, t_final, h0=0.01, tol=tol_bonus, h_min=1e-6, h_max=1.0
)

print("Running PI-controller solver (tol=1e-4) …")
t_PI, y_PI, h_hist_PI, stats_PI = adaptive_implicit_euler_pi(
    V0, t0, t_final, h0=0.01, tol=tol_bonus, h_min=1e-6, h_max=1.0
)

V_ref_P  = analytical_V(t_P)
V_ref_PI = analytical_V(t_PI)

print(f"\n{'Method':<20} {'Accepted':>10} {'Rejected':>10} "
      f"{'NR Iters':>12} {'Max Error':>14}")
print("-" * 68)
print(f"{'P-controller':<20} {stats_P['accepted_steps']:>10} "
      f"{stats_P['rejected_steps']:>10} {stats_P['function_evals']:>12} "
      f"{np.max(np.abs(y_P - V_ref_P)):>14.4e}")
print(f"{'PI-controller':<20} {stats_PI['accepted_steps']:>10} "
      f"{stats_PI['rejected_steps']:>10} {stats_PI['function_evals']:>12} "
      f"{np.max(np.abs(y_PI - V_ref_PI)):>14.4e}")


### Bonus 3: Work-Precision Diagram

In [ ]:
def crank_nicolson_step(y_n, t_n, h):
    
    t_next = t_n + h
    f_n = rc_rhs(t_n, y_n)
    g_prime = 1.0 + 0.5 * h * lam   # constant Jacobian for this linear ODE
    V = y_n
    iters = 0
    for _ in range(50):
        g = V - y_n - 0.5 * h * (f_n + rc_rhs(t_next, V))
        if abs(g) < 1e-12:
            break
        V -= g / g_prime
        iters += 1
    return V, iters + 1

def adaptive_implicit_euler_rk2(y0, t0, t_final, h0, tol,
                                 h_min=1e-6, h_max=1.0, safety=0.9,
                                 max_steps=100_000):
    
    t = [t0];  y = [y0]
    h_history = [];  error_history = []

    stats = {'accepted_steps': 0, 'rejected_steps': 0, 'function_evals': 0}

    h = h0;  n_steps = 0

    while t[-1] < t_final and n_steps < max_steps:
        t_n = t[-1];  y_n = y[-1]
        h = min(h, t_final - t_n)
        if h < h_min:
            h = h_min

        y_ie, iters_ie = implicit_euler_step(y_n, t_n, t_n + h, h)
        y_cn, iters_cn = crank_nicolson_step(y_n, t_n, h)

        stats['function_evals'] += iters_ie + iters_cn
        error = abs(y_cn - y_ie)

        if error < 1e-14:
            h_new = min(2.0 * h, h_max)
        else:
            h_new = safety * h * (tol / error) ** (1.0 / 3.0)
        h_new = max(h_min, min(h_max, h_new))
        h_new = max(0.2 * h, min(5.0 * h, h_new))

        if error < tol:
            t.append(t_n + h)
            y.append(y_cn)   # use higher-order CN result
            h_history.append(h)
            error_history.append(error)
            stats['accepted_steps'] += 1
        else:
            stats['rejected_steps'] += 1

        h = h_new;  n_steps += 1

    stats['error_history'] = np.array(error_history)
    return np.array(t), np.array(y), np.array(h_history), stats


In [ ]:
tolerances = [1e-2, 3e-3, 1e-3, 3e-4, 1e-4, 3e-5, 1e-5, 3e-6, 1e-6]

wp_sd  = []   # step-doubling
wp_rk2 = []   # embedded RK2
wp_fix = []   # fixed step

for tol_i in tolerances:
    t_sd_i, y_sd_i, _, s_sd = adaptive_implicit_euler(
        V0, t0, t_final, h0=0.01, tol=tol_i, h_min=1e-8, h_max=1.0, safety=0.9
    )
    err_sd = np.max(np.abs(y_sd_i - analytical_V(t_sd_i)))
    wp_sd.append((s_sd['function_evals'], err_sd))

    t_r2_i, y_r2_i, _, s_r2 = adaptive_implicit_euler_rk2(
        V0, t0, t_final, h0=0.01, tol=tol_i, h_min=1e-8, h_max=1.0, safety=0.9
    )
    err_rk2 = np.max(np.abs(y_r2_i - analytical_V(t_r2_i)))
    wp_rk2.append((s_r2['function_evals'], err_rk2))

h_fixed_range = [2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]
for h_fi in h_fixed_range:
    t_fi, y_fi, ni_fi = euler_backward_fixed(V0, t0, t_final, h_fi)
    err_fi = np.max(np.abs(y_fi - analytical_V(t_fi)))
    if err_fi > 0:
        wp_fix.append((ni_fi, err_fi))

wp_fix.sort()                    
wp_sd.sort();  wp_rk2.sort()

fig, ax = plt.subplots(figsize=(4.5, 3.1))

evals_sd,  errs_sd  = zip(*wp_sd)
evals_rk2, errs_rk2 = zip(*wp_rk2)
evals_fix, errs_fix  = zip(*wp_fix)

ax.loglog(evals_fix, errs_fix,  'g^--', markersize=8,  lw=1.5, label='Fixed step size (Euler Backward)')
ax.loglog(evals_sd,  errs_sd,   'bo-',  markersize=9,  lw=2,   label='Adaptive — step-doubling')
ax.loglog(evals_rk2, errs_rk2,  'rs-',  markersize=9,  lw=2,   label='Adaptive — embedded Euler/CN')

for (ev, er), tol_i in zip(wp_sd, tolerances):
    if tol_i in [1e-2, 1e-4, 1e-6]:
        ax.annotate(f'tol={tol_i:.0e}', xy=(ev, er),
                    xytext=(ev * 1.3, er * 2.5), fontsize=8,
                    arrowprops=dict(arrowstyle='->', lw=0.8))

ax.set_xlabel('Total NR Function Evaluations  (proxy for CPU work)', fontsize=12)
ax.set_ylabel('Max Absolute Error  [V]', fontsize=12)
ax.set_title('Bonus 3 — Work-Precision Diagram\n'
             'RC Circuit: Adaptive vs. Fixed Step Strategies', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

print("\nWork-Precision Summary at comparable accuracy (~1e-4 error):")
print(f"{'Strategy':<35} {'Work (NR iters)':>18} {'Max Error':>14}")
print("-" * 68)

for (ev, er) in wp_fix:
    if er < 2e-4:
        print(f"{'Fixed step':35} {ev:>18d} {er:>14.4e}")
        break

idx = tolerances.index(1e-4)
ev_sd,  er_sd  = wp_sd [idx]
ev_rk2, er_rk2 = wp_rk2[idx]
print(f"{'Adaptive step-doubling (tol=1e-4)':<35} {int(ev_sd):>18} {er_sd:>14.4e}")
print(f"{'Adaptive embedded RK2  (tol=1e-4)':<35} {int(ev_rk2):>18} {er_rk2:>14.4e}")
print("\nKey insight: adaptive solvers achieve the same accuracy with "
      "significantly fewer function evaluations than fixed-step methods.")
